# Beca 18 RAG Chatbot
**HW03 · Applied Data Science**

RAG chatbot that answers questions about Beca 18 regulations (PRONABEC RDE 033-2026)  
using Gemini embeddings, ChromaDB for vector storage, and grounded generation.

---
## Section 1 — Install Dependencies

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

# In Colab: clone the repo so requirements.txt and all project files are available,
# then change into it so every subsequent cell uses the right working directory.
# Locally: requirements.txt is already present — skip the clone.
if IN_COLAB:
    repo_dir = "beca18-rag-chatbot"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "https://github.com/Erics-20/beca18-rag-chatbot.git"],
            check=True,
        )
    os.chdir(repo_dir)
    print(f"Working directory set to: {os.getcwd()}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"],
    check=True,
)
print("Installation complete. If running in Colab, restart the session now (Runtime > Restart session).")

---
## Section 2 — Load API Key from `.env`

**How to set up your `.env` file:**

Create a file named `.env` in the project root with this single line:
```
GEMINI_API_KEY=your_actual_key_here
```

**In Google Colab:** run the cell below — it will prompt you to upload your `.env` file.  
**Locally (Jupyter):** place `.env` in the same folder as this notebook and run the cell.

> The key is never stored in the notebook. The `.env` file is listed in `.gitignore`.

In [ ]:
import sys
import os
from dotenv import load_dotenv

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Running in Colab — please upload your .env file when prompted.")
    uploaded = files.upload()          # opens the file-picker dialog
    if ".env" not in uploaded:
        raise FileNotFoundError("No .env file was uploaded. Please upload a file named exactly '.env'.")
    # Write the uploaded bytes to disk so dotenv can read it
    with open(".env", "wb") as f:
        f.write(uploaded[".env"])
    print(".env file saved to the Colab working directory.")
else:
    print("Running locally — expecting .env in the notebook directory.")

load_dotenv(dotenv_path=".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY not found. "
        "Make sure your .env file contains: GEMINI_API_KEY=your_key_here"
    )

print(f"GEMINI_API_KEY loaded successfully (starts with: {GEMINI_API_KEY[:6]}...).")

---
## Section 3 — Confirm Environment

Print the installed package versions to verify the environment is correctly set up.

In [ ]:
import importlib.metadata
import platform

packages = [
    "pypdf",
    "tiktoken",
    "langchain-text-splitters",
    "google-genai",
    "chromadb",
    "ipywidgets",
    "tqdm",
    "python-dotenv",
]

print(f"Python            : {platform.python_version()}")
print("-" * 40)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        status = "OK"
    except importlib.metadata.PackageNotFoundError:
        version = "NOT INSTALLED"
        status = "MISSING"
    print(f"{pkg:<30} {version:<12} [{status}]")

print("-" * 40)
print("Environment check complete.")

---
## Section 4 — Extract and Clean PDF Text

**Before running this cell**, upload `beca18_reglamento.pdf` to your Google Drive (anywhere in My Drive).  
The cell will mount Drive, find the file automatically, and copy it into the `data/` folder.

**Cleaning steps applied per page:**
1. Repeated lines detected across pages (headers/footers) are removed.
2. Isolated line breaks (single `\n` inside a paragraph) are collapsed into spaces.
3. Multiple consecutive spaces/tabs are collapsed into one.
4. A `[PAGE N]` marker is prepended to each page.

In [ ]:
import re
import shutil
from pathlib import Path
from pypdf import PdfReader

# ── Configuration ──────────────────────────────────────────────────────────────
HEADER_FOOTER_LINES = 2    # lines to inspect at top/bottom of each page
REPEAT_THRESHOLD    = 0.4  # a line appearing on ≥40 % of pages → header/footer
PDF_NAME            = "beca18_reglamento.pdf"

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# ── In Colab: mount Google Drive and copy the PDF into data/ ──────────────────
if IN_COLAB and not list(data_dir.glob("*.pdf")):
    from google.colab import drive
    drive.mount("/content/drive")
    matches = list(Path("/content/drive/MyDrive").rglob(PDF_NAME))
    if not matches:
        raise FileNotFoundError(
            f"{PDF_NAME} not found anywhere in your Google Drive. "
            "Upload it to Drive and re-run this cell."
        )
    shutil.copy(matches[0], data_dir / PDF_NAME)
    print(f"Copied from Drive: {matches[0]}")

# ── Resolve PDF path ───────────────────────────────────────────────────────────
pdfs = list(data_dir.glob("*.pdf"))
if len(pdfs) == 0:
    raise FileNotFoundError(
        f"No PDF found in data/. Add {PDF_NAME} to that folder and re-run."
    )
PDF_PATH = data_dir / PDF_NAME if (data_dir / PDF_NAME).exists() else pdfs[0]
print(f"PDF path: {PDF_PATH}")


# ── Helper: detect repeated header/footer lines ────────────────────────────────
def detect_repeated_lines(pages_raw: list, n_edge: int, threshold: float) -> set:
    from collections import Counter
    candidates = Counter()
    for text in pages_raw:
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        edge = lines[:n_edge] + lines[-n_edge:]
        for line in set(edge):
            candidates[line] += 1
    total = len(pages_raw)
    return {line for line, count in candidates.items() if count / total >= threshold}


# ── Helper: clean one page ─────────────────────────────────────────────────────
def clean_page(text: str, repeated: set) -> str:
    lines = [l for l in text.splitlines() if l.strip() not in repeated]
    text = "\n".join(lines)
    # Collapse isolated line breaks (single \n → space, keep paragraph breaks \n\n)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse multiple spaces/tabs into one
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


# ── Extract raw text page by page ─────────────────────────────────────────────
reader    = PdfReader(PDF_PATH)
pages_raw = [page.extract_text() or "" for page in reader.pages]
print(f"Pages extracted: {len(pages_raw)}")

# ── Detect repeated lines (headers / footers) ─────────────────────────────────
repeated = detect_repeated_lines(pages_raw, HEADER_FOOTER_LINES, REPEAT_THRESHOLD)
if repeated:
    print(f"\nDetected {len(repeated)} repeated header/footer line(s) — will be removed:")
    for line in sorted(repeated):
        print(f"  • {line!r}")
else:
    print("\nNo repeated headers/footers detected.")

# ── Clean pages and prepend [PAGE N] markers ──────────────────────────────────
pages_clean = []
for i, raw in enumerate(pages_raw, start=1):
    cleaned = clean_page(raw, repeated)
    pages_clean.append(f"[PAGE {i}]\n{cleaned}")

full_text = "\n\n".join(pages_clean)

# ── Stats ──────────────────────────────────────────────────────────────────────
char_count = len(full_text)
word_count = len(full_text.split())

print(f"\n{'─'*40}")
print(f"Pages  : {len(pages_clean):>10,}")
print(f"Chars  : {char_count:>10,}")
print(f"Words  : {word_count:>10,}")
print(f"{'─'*40}")

---
## Section 5 — Tokenisation & Chunking

### Why 400 tokens with 60-token overlap?

The Gemini embedding model (`text-embedding-004`) accepts up to **8,192 tokens** per input.  
Choosing `chunk_size = 400` keeps each chunk at roughly **5 % of that limit**, which gives three concrete benefits:

| Property | Value | Rationale |
|---|---|---|
| Chunk size | 400 tokens ≈ 300 words | Large enough to hold a complete article clause or paragraph; small enough to retrieve a precise answer |
| Overlap | 60 tokens ≈ 15 % of chunk | Prevents a sentence that straddles two chunk boundaries from being lost in both |
| Headroom | 8,192 − 400 = 7,792 tokens free | Zero risk of truncation; leaves room for future prompt augmentation |

Larger chunks (e.g. 1,000+ tokens) would embed more context but hurt **retrieval precision** — the model scores the whole chunk, so relevant sentences buried inside long chunks rank lower.  
Smaller chunks (< 200 tokens) risk splitting mid-sentence and losing the grammatical subject of a regulation clause, which is fatal for a legal Q&A use case.  
**400 / 60 is the sweet spot for a dense regulatory document in Spanish.**

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Token count ────────────────────────────────────────────────────────────────
enc          = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(full_text))

print(f"Total tokens (cl100k_base): {total_tokens:,}")
print(f"Embedding limit            : 8,192 tokens")
print(f"Full doc / limit ratio     : {total_tokens / 8192:.1f}x  →  chunking required\n")

# ── Chunking parameters ────────────────────────────────────────────────────────
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60

# Use tiktoken to measure length in tokens so chunk_size means exactly 400 tokens.
def token_len(text: str) -> int:
    return len(enc.encode(text))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
    length_function=token_len,
)

# ── Metadata attached to every chunk ──────────────────────────────────────────
metadata = {
    "document": PDF_PATH.name,
    "topic"   : "Beca 18 – Reglamento de Beneficiarios (PRONABEC RDE 033-2026)",
    "language": "es",
}

chunks = splitter.create_documents([full_text], metadatas=[metadata])

# ── Stats ──────────────────────────────────────────────────────────────────────
total_chunks  = len(chunks)
avg_char_len  = sum(len(c.page_content) for c in chunks) / total_chunks
avg_token_len = sum(token_len(c.page_content) for c in chunks) / total_chunks

print(f"{'─'*40}")
print(f"Total chunks     : {total_chunks:>8,}")
print(f"Avg length (chars): {avg_char_len:>7,.0f}")
print(f"Avg length (tokens): {avg_token_len:>6,.1f}")
print(f"{'─'*40}")

# Preview the first chunk
print(f"\n── Chunk 0 preview ──")
print(f"Metadata : {chunks[0].metadata}")
print(f"Content  : {chunks[0].page_content[:300]}...")

---
## Section 6 — Gemini Embedding Functions

### Two task types, two functions

Gemini's `gemini-embedding-001` model is trained with **asymmetric retrieval**: the document and query embedding spaces are optimised differently so that a short question lands close to a long passage that answers it — even though they share few words.

| Function | Task type | Used when |
|---|---|---|
| `embed_documents(texts)` | `RETRIEVAL_DOCUMENT` | Indexing chunks into ChromaDB |
| `embed_query(text)` | `RETRIEVAL_QUERY` | Embedding the user's question at query time |

Using the wrong task type degrades retrieval quality, so both task types are made explicit.

### Exponential backoff with jitter

The Gemini free tier allows ≈ 60 requests per minute. Under load, the API returns HTTP **429 (Resource Exhausted)**. The retry strategy is:

- Base wait: **1 s → 2 s → 4 s → 8 s → 16 s** (doubles each attempt)  
- **Jitter** (`+0–0.5 s` random): prevents multiple concurrent callers from retrying at exactly the same moment (thundering-herd problem)  
- After **5 failed attempts** the exception is re-raised so the notebook fails loudly rather than silently

### Batching

`embed_documents` sends texts in batches of up to **100** per API call and inserts a **1-second pause** between batches, keeping throughput safely below the rate limit even on the free tier.

In [ ]:
import time
import random
from tqdm.notebook import tqdm
from google import genai
from google.genai import types

# ── Gemini client ──────────────────────────────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

EMBED_MODEL = "models/gemini-embedding-001"
EMBED_DIM   = 768
BATCH_SIZE  = 100   # texts per API call


# ── Core API call with exponential backoff + jitter ───────────────────────────
def _call_embed_api(contents: list, task_type: str, max_retries: int = 5) -> list:
    wait = 1.0
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBED_MODEL,
                contents=contents,
                config=types.EmbedContentConfig(
                    task_type=task_type,
                    output_dimensionality=EMBED_DIM,
                ),
            )
            return [e.values for e in response.embeddings]
        except Exception as exc:
            msg = str(exc).lower()
            is_rate_limit = "429" in msg or "quota" in msg or "rate" in msg
            if is_rate_limit and attempt < max_retries - 1:
                delay = wait + random.uniform(0, 0.5)
                print(f"  Rate limit — waiting {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
                wait *= 2
            else:
                raise


# ── Public functions ───────────────────────────────────────────────────────────
def embed_documents(texts: list) -> list:
    """Embed texts for indexing using RETRIEVAL_DOCUMENT task type.
    Batches requests and pauses between batches to respect the free-tier RPM limit.
    """
    all_embeddings = []
    batch_starts = list(range(0, len(texts), BATCH_SIZE))
    for i in tqdm(batch_starts, desc="Embedding documents", unit="batch"):
        batch = texts[i : i + BATCH_SIZE]
        all_embeddings.extend(_call_embed_api(batch, task_type="RETRIEVAL_DOCUMENT"))
        if i + BATCH_SIZE < len(texts):
            time.sleep(1.0)   # ~60 RPM guard
    return all_embeddings


def embed_query(text: str) -> list:
    """Embed a single query string using RETRIEVAL_QUERY task type."""
    return _call_embed_api([text], task_type="RETRIEVAL_QUERY")[0]


# ── Smoke test ─────────────────────────────────────────────────────────────────
print("Running smoke test…")

query_vec = embed_query("¿Qué requisitos necesito para postular a Beca 18?")
print(f"embed_query     → dim: {len(query_vec)},  sample: {[round(v, 6) for v in query_vec[:4]]}")

doc_vecs = embed_documents([chunks[0].page_content, chunks[1].page_content])
print(f"embed_documents → {len(doc_vecs)} vectors, dim: {len(doc_vecs[0])}")

print("\nEmbedding functions ready ✓")